# UART loopback test

Sends a 128x128 grayscale image to the board over UART, reads it back after the board has stored it in block RAM, and reports any pixel that differs.

1. Program the Arty S7 with the loopback design and set `PORT` to the board's serial port.
2. Run the first cell, then flip SW0 (receive) on the board and run the send cell.
3. When LED2 lights, flip SW1 (transmit) and run the receive and compare cells.

In [ ]:
import cv2
import numpy as np
import serial
import time

PORT = "COM4"      # serial port of the board, e.g. COM4 on Windows or /dev/ttyUSB1 on Linux
BAUD = 115200
SIZE = 128

image = cv2.imread("images/testpattern.png", cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (SIZE, SIZE))
cv2.imwrite("images/original.bmp", image)
transmit_data = np.array(image, dtype=np.uint8)
print(transmit_data.size, "bytes to send")

In [ ]:
link = serial.Serial(PORT, BAUD, timeout=1)
link.write(transmit_data.tobytes())

In [ ]:
# wait for the echo's first byte (the test pattern starts with a non-zero pixel), then read the rest
result = []
while True:
    value = int.from_bytes(link.read(), byteorder="big")
    if value != 0:
        result.append(value)
        break

for i in range(SIZE * SIZE - 1):
    result.append(int.from_bytes(link.read(), byteorder="big"))
print(len(result), "bytes received")

In [ ]:
received = np.array(result, dtype=np.uint8).reshape(SIZE, SIZE)
mismatches = np.argwhere(received != image)
for i, j in mismatches[:20]:
    print(f"({i}, {j}) sent {image[i, j]} received {received[i, j]}")
print(f"{len(mismatches)} of {SIZE * SIZE} pixels differ")
cv2.imwrite("images/result.bmp", received)